# Optimitzar llegendes sense reentrenar les columnes

Compara el baseline, l’especialista, sis unions de fragments, marcs i combinacions. Utilitza els pesos existents. Els 10 documents són desenvolupament, no un test independent.

Puja a Drive el `colab_bundle.zip` actualitzat i `legend_specialist.zip`. Mantén els JSON base del primer entrenament. Els dos entrenaments de bandes són opcionals i consecutius.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from datetime import datetime
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/subsurface-bboxes')
BUNDLE = DRIVE_ROOT / 'colab_bundle.zip'
LEGEND_ARCHIVE = DRIVE_ROOT / 'legend_specialist.zip'
BASE_RUN = DRIVE_ROOT / 'runs' / '20261107-151404'
BASE_PREDICTIONS = BASE_RUN / 'validation' / 'predictions'
WORKDIR = Path('/content/legend-optimization')
RUN = DRIVE_ROOT / 'legend-optimization' / datetime.now().strftime('%Y%m%d-%H%M%S')
RUN_BAND_TRAINING = False  # Activa-ho per provar els refinadors 1024/1536.
RUN.mkdir(parents=True, exist_ok=True)
assert BUNDLE.is_file() and LEGEND_ARCHIVE.is_file()
assert BASE_PREDICTIONS.is_dir()


In [ ]:
import importlib, json, os, shutil, subprocess, sys, zipfile
WORKDIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    for member in archive.infolist():
        if not (WORKDIR / member.filename).resolve().is_relative_to(WORKDIR.resolve()):
            raise ValueError('Ruta invàlida al ZIP')
    archive.extractall(WORKDIR)
assert (WORKDIR / 'bbox_detector/legend_benchmark.py').is_file(), 'Cal regenerar colab_bundle.zip amb el codi nou'
os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))
for name in list(sys.modules):
    if name == 'bbox_detector' or name.startswith('bbox_detector.'):
        del sys.modules[name]
importlib.invalidate_caches()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
LEGEND_WEIGHTS = WORKDIR / 'legend_best.pt'
# Importa exclusivament els pesos; no executa ni substitueix el codi de l’arxiu.
with zipfile.ZipFile(LEGEND_ARCHIVE) as archive:
    with archive.open('legend_best.pt') as source, LEGEND_WEIGHTS.open('wb') as target:
        shutil.copyfileobj(source, target)
import torch
DEVICE = '0' if torch.cuda.is_available() else 'cpu'
print('Dispositiu:', DEVICE, 'Resultats:', RUN)


## 1. Split i deteccions de l’especialista

Les anotacions només intervenen en preparació i puntuació. El predictor rep la imatge i els JSON de columnes. Les prediccions inclouen tots els grups candidats per comparar geometria sense repetir YOLO.


In [ ]:
from bbox_detector.data import prepare_dataset
from bbox_detector.evaluation import evaluate_manifest
from bbox_detector.legend_inference import make_cached_legend_fusion_predictor
manifest = prepare_dataset(WORKDIR / 'dataset', RUN / 'audit', materialize=False)
MANIFEST = RUN / 'audit/manifest.json'
coarse_predictor = make_cached_legend_fusion_predictor(
    BASE_PREDICTIONS, LEGEND_WEIGHTS, imgsz=2048, confidence=.05, device=DEVICE)
coarse = evaluate_manifest(MANIFEST, None, RUN / 'coarse', predictor=coarse_predictor,
                           imgsz=2048, conf=.05, device=DEVICE, save_previews=False)
print(json.dumps(manifest['summary'], indent=2))


## 2. Benchmark i selecció

Prioritat: nombre de llegendes amb IoU ≥ 0,8; després IoU mitjana. Els empats conserven l’estratègia anterior. Les columnes són les mateixes a totes les variants. La revisió dels 51 formats es troba a `docs/legend_review.json` i només s’utilitza per desglossar mètriques.


In [ ]:
from bbox_detector.legend_benchmark import benchmark_legend_policies
comparison = benchmark_legend_policies(
    MANIFEST, BASE_PREDICTIONS, RUN / 'coarse/predictions', RUN / 'benchmark',
    formats_path=WORKDIR / 'docs/legend_review.json', save_previews=True)
POLICY = RUN / 'benchmark/selected_policy.json'
import pandas as pd
rows = [{'estrategia': name, 'IoU': s['iou_mean'], 'IoU_08': s['iou_ge_0_8'],
         'IoU_09': s['iou_ge_0_9'], 'absents': s['missing']}
        for name, s in comparison['results'].items()]
display(pd.DataFrame(rows).sort_values(['IoU_08', 'IoU'], ascending=False))
print('Seleccionada:', comparison['selected_strategy'])
print('Regressions:', comparison['regressions_vs_previous'])
print('Comparacions:', RUN / 'benchmark/selected/visual_review/index.html')


## 3. Temps real de l’especialista + reconstrucció

Aquest pas torna a executar YOLO i la geometria. El temps exclou el detector general de columnes, perquè es reutilitzen els seus JSON, i exclou la càrrega inicial dels pesos.


In [ ]:
from bbox_detector.legend_pipeline import LegendPolicy, apply_legend_policy
from bbox_detector.legend_inference import UltralyticsLegendDetector
policy = LegendPolicy.load(POLICY)
detector = UltralyticsLegendDetector(LEGEND_WEIGHTS)
def live_predictor(image_path, **_kwargs):
    base = json.loads((RUN / 'benchmark/baseline/predictions' / f'{Path(image_path).stem}.json').read_text())
    details = {}
    prediction = apply_legend_policy(image_path, base, policy, detector=detector,
                                     device=DEVICE, diagnostics=details)
    return {**prediction, 'diagnostics': details}
live_report = evaluate_manifest(MANIFEST, None, RUN / 'live', predictor=live_predictor,
                                imgsz=2048, conf=.05, device=DEVICE, save_previews=False)
print(json.dumps(live_report['summary'], indent=2))


## 4. Refinadors opcionals sobre bandes

`RUN_BAND_TRAINING=True` prepara un únic dataset rectangular i entrena dues inicialitzacions independents de YOLO11s a 1024 i 1536. Mateix split, seed, AdamW, batch, augmentacions i parada anticipada. La validació final obté les bandes de prediccions, mai de les anotacions. Un refí que empitjora el resultat es descarta.


In [ ]:
band_comparison = None
if RUN_BAND_TRAINING:
    if DEVICE == 'cpu':
        raise RuntimeError('Activa una GPU per executar els dos entrenaments opcionals')
    from bbox_detector.legend_data import prepare_legend_dataset
    from bbox_detector.legend_experiments import train_band_experiments
    prepare_legend_dataset(MANIFEST, WORKDIR / 'band_dataset', imgsz=1536,
                           crop_mode='band', train_variants=4)
    band_comparison = train_band_experiments(
        WORKDIR / 'band_dataset/dataset.yaml', MANIFEST, BASE_PREDICTIONS,
        RUN / 'coarse/predictions', POLICY, RUN / 'band_runs', device=DEVICE)
    print(json.dumps(band_comparison, indent=2))
else:
    print('Es conserven els pesos existents; entrenaments de bandes no executats.')


## 5. Predicció i exportació

El paquet conté codi, configuració, pesos existents i informes. No inclou TIFFs. Si es selecciona un refinador opcional, també inclou els seus pesos; utilitza els flags `--legend-band-weights` i `--legend-band-imgsz` indicats a la selecció exportada.


In [ ]:
example = next(r for r in manifest['records'] if r['split'] == 'val')
command = [sys.executable, 'predict_bboxes.py', '--image', example['image_path'],
           '--base-prediction', str(RUN / 'benchmark/baseline/predictions' / f"{Path(example['image_name']).stem}.json"),
           '--legend-weights', str(LEGEND_WEIGHTS), '--legend-config', str(POLICY),
           '--output', str(RUN / 'example_prediction.json'), '--device', DEVICE]
if band_comparison and band_comparison['band_weights']:
    command += ['--legend-band-weights', band_comparison['band_weights'],
                '--legend-band-imgsz', str(band_comparison['band_imgsz'])]
subprocess.run(command, check=True)
EXPORT = RUN / 'export'
EXPORT.mkdir(exist_ok=True)
for folder in ('bbox_detector', 'tools', 'docs'):
    shutil.copytree(WORKDIR / folder, EXPORT / folder, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns('__pycache__'))
for name in ('predict_bboxes.py', 'evaluate_bboxes.py', 'requirements.txt'):
    shutil.copy2(WORKDIR / name, EXPORT / name)
if (WORKDIR / 'build_manifest.json').is_file():
    shutil.copy2(WORKDIR / 'build_manifest.json', EXPORT / 'build_manifest.json')
shutil.copy2(LEGEND_WEIGHTS, EXPORT / 'legend_best.pt')
shutil.copy2(POLICY, EXPORT / 'legend_policy.json')
shutil.copy2(RUN / 'benchmark/comparison.json', EXPORT / 'comparison.json')
shutil.copy2(RUN / 'live/report.json', EXPORT / 'validation_report.json')
shutil.copytree(RUN / 'live/predictions', EXPORT / 'predictions', dirs_exist_ok=True)
if band_comparison:
    (EXPORT / 'band_selection.json').write_text(json.dumps(band_comparison, indent=2))
    if band_comparison['band_weights']:
        shutil.copy2(band_comparison['band_weights'], EXPORT / 'legend_band_best.pt')
archive = shutil.make_archive(str(RUN / 'legend_optimized'), 'zip', EXPORT)
print('Paquet:', archive)
